In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (orden recomendado): Partición temporal (train/validation/test) y normalización.
Entrada: datos codificados (salida del Código 3) en las carpetas:
    - encoded/ml/by_station/
    - encoded/ml/global/
    - encoded/dl/by_station/
    - encoded/dl/global/
Salida: conjuntos particionados y normalizados (para DL) en:
    windows_partitioned/
        by_station/
            ml/ (sin normalizar) y dl/ (normalizados)
        global/
            ml/ y dl/
Se guardan también los escaladores para poder invertir la normalización.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_ML_BY_STATION = os.path.join(BASE_DIR, "encoded", "ml", "by_station")
INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_BY_STATION = os.path.join(BASE_DIR, "encoded", "dl", "by_station")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_DIR, "by_station", "ml")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_DIR, "by_station", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros de ventanas
WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

# Fechas de corte (según especificación)
TRAIN_END = '2023-12-31 23:00:00'
VAL_START = '2024-01-01 00:00:00'
VAL_END = '2024-12-31 23:00:00'
TEST_START = '2025-01-01 00:00:00'
# TEST_END = '2025-12-31 23:00:00'  # se asume hasta el final

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def create_windows_with_timestamps(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    """
    Similar a create_windows_from_df, pero además devuelve los timestamps de inicio de cada ventana.
    Retorna:
        X: (n_windows, window_in, n_features)
        y: (n_windows, window_out)
        timestamps: lista de datetime de inicio de cada ventana
        feature_names: lista de nombres de las columnas
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index()
    data = df_sorted.values
    n_features = data.shape[1]
    feature_names = df_sorted.columns.tolist()
    timestamps = []

    X_list = []
    y_list = []

    n = len(df_sorted)
    for i in range(window_in, n - window_out):
        # Ventana de entrada: [i-window_in , i)
        in_data = data[i - window_in:i, :]                # shape (window_in, n_features)
        # Ventana de salida (solo target): [i , i+window_out)
        out_data = df_sorted[target_col].iloc[i:i + window_out].values  # shape (window_out,)
        X_list.append(in_data)
        y_list.append(out_data)
        timestamps.append(df_sorted.index[i])  # timestamp del inicio de la predicción

    if len(X_list) == 0:
        return None, None, None, feature_names

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, timestamps, feature_names

def split_by_timestamps(X, y, timestamps, train_end, val_start, val_end, test_start):
    """
    Divide las ventanas según las fechas de los timestamps.
    Retorna:
        (X_train, y_train), (X_val, y_val), (X_test, y_test)
    """
    # Convertir timestamps a array de pandas Timestamp para comparación fácil
    ts = pd.to_datetime(timestamps)
    train_mask = ts <= pd.to_datetime(train_end)
    val_mask = (ts >= pd.to_datetime(val_start)) & (ts <= pd.to_datetime(val_end))
    test_mask = ts >= pd.to_datetime(test_start)

    # Verificar que no haya solapamiento
    assert not (train_mask & val_mask).any(), "Solapamiento train/val"
    assert not (train_mask & test_mask).any(), "Solapamiento train/test"
    assert not (val_mask & test_mask).any(), "Solapamiento val/test"

    X_train = X[train_mask]
    y_train = y[train_mask]
    X_val = X[val_mask]
    y_val = y[val_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test, feature_names):
    """
    Aplica MinMaxScaler a X e y (por separado). Ajusta con train y transforma todos.
    Retorna:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y
    Nota: X tiene forma (n, window_in, n_features). Para escalar, aplanamos las ventanas.
    """
    n_train, win_in, n_feat = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    # Aplanar a 2D: (n * win_in, n_feat)
    X_train_flat = X_train.reshape(-1, n_feat)
    X_val_flat = X_val.reshape(-1, n_feat)
    X_test_flat = X_test.reshape(-1, n_feat)

    # Escalar X
    scaler_X = MinMaxScaler()
    X_train_scaled_flat = scaler_X.fit_transform(X_train_flat)
    X_val_scaled_flat = scaler_X.transform(X_val_flat)
    X_test_scaled_flat = scaler_X.transform(X_test_flat)

    # Reconstruir forma 3D
    X_train_sc = X_train_scaled_flat.reshape(n_train, win_in, n_feat)
    X_val_sc = X_val_scaled_flat.reshape(n_val, win_in, n_feat)
    X_test_sc = X_test_scaled_flat.reshape(n_test, win_in, n_feat)

    # Escalar y (target)
    scaler_y = MinMaxScaler()
    # y es (n, window_out), aplanamos a 2D: (n, window_out) -> (n * window_out, 1) para scaler
    y_train_flat = y_train.reshape(-1, 1)
    y_val_flat = y_val.reshape(-1, 1)
    y_test_flat = y_test.reshape(-1, 1)

    scaler_y.fit(y_train_flat)  # ajustar con train
    y_train_scaled_flat = scaler_y.transform(y_train_flat)
    y_val_scaled_flat = scaler_y.transform(y_val_flat)
    y_test_scaled_flat = scaler_y.transform(y_test_flat)

    # Reconstruir a (n, window_out)
    y_train_sc = y_train_scaled_flat.reshape(n_train, win_out)
    y_val_sc = y_val_scaled_flat.reshape(n_val, win_out)
    y_test_sc = y_test_scaled_flat.reshape(n_test, win_out)

    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y

def save_split(output_dir, name, X_train, y_train, X_val, y_val, X_test, y_test,
               feature_names, scaler_X=None, scaler_y=None):
    """
    Guarda los conjuntos en la carpeta output_dir/name/ (si name no es vacío, se crea subcarpeta)
    """
    if name:
        save_dir = os.path.join(output_dir, name)
    else:
        save_dir = output_dir
    os.makedirs(save_dir, exist_ok=True)

    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)

    with open(os.path.join(save_dir, "features.json"), 'w') as f:
        json.dump(feature_names, f, indent=2)

    if scaler_X is not None:
        with open(os.path.join(save_dir, "scaler_X.pkl"), 'wb') as f:
            pickle.dump(scaler_X, f)
    if scaler_y is not None:
        with open(os.path.join(save_dir, "scaler_y.pkl"), 'wb') as f:
            pickle.dump(scaler_y, f)

def process_station(csv_path_ml, csv_path_dl, station_name, output_ml_dir, output_dl_dir):
    """
    Procesa una estación: genera ventanas, particiona y guarda versiones ML (sin escalar) y DL (escaladas).
    """
    print(f"  Procesando estación: {station_name}")

    # Cargar datos ML y DL (deben tener el mismo índice)
    df_ml = pd.read_csv(csv_path_ml, index_col=0, parse_dates=True)
    df_dl = pd.read_csv(csv_path_dl, index_col=0, parse_dates=True)

    # Verificar que los índices coinciden
    if not df_ml.index.equals(df_dl.index):
        print(f"    Advertencia: los índices ML y DL no coinciden para {station_name}. Se usará ML para features y DL para target?")
        # Forzar alineación (peligroso, mejor detenerse)
        # Por simplicidad, asumimos que son iguales; si no, se podría hacer un merge.
        # En este caso, lanzamos error para que el usuario verifique.
        raise ValueError(f"Los índices de ML y DL no coinciden para {station_name}. Asegúrate de que los archivos están alineados.")

    # Generar ventanas con timestamps (usamos df_ml, que tiene las mismas filas que df_dl)
    X, y, timestamps, feature_names = create_windows_with_timestamps(df_ml)
    if X is None:
        print(f"    No se generaron ventanas para {station_name}")
        return

    # Partición temporal
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_by_timestamps(
        X, y, timestamps, TRAIN_END, VAL_START, VAL_END, TEST_START
    )

    print(f"    Ventanas: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

    # Guardar versión ML (sin normalizar)
    save_split(output_ml_dir, station_name,
               X_train, y_train, X_val, y_val, X_test, y_test,
               feature_names, scaler_X=None, scaler_y=None)

    # Para DL: normalizar (usamos los mismos X, y pero escalados)
    # Nota: la normalización se aplica sobre los datos de entrenamiento y luego se transforman val/test.
    if len(X_train) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train, X_val, X_test, y_train, y_val, y_test, feature_names
        )
        save_split(output_dl_dir, station_name,
                   X_train_sc, y_train_sc, X_val_sc, y_val_sc, X_test_sc, y_test_sc,
                   feature_names, scaler_X, scaler_y)
    else:
        print(f"    No hay datos de entrenamiento para {station_name}, se omite DL.")

def process_global(ml_dir, dl_dir, output_ml_dir, output_dl_dir):
    """
    Procesa el caso global: combina todas las estaciones, genera ventanas, particiona y normaliza.
    ml_dir: carpeta con archivos CSV para ML (one-hot)
    dl_dir: carpeta con archivos CSV para DL (enteros)
    """
    print("  Procesando conjunto global...")
    ml_files = list(Path(ml_dir).glob("*.csv"))
    if not ml_files:
        print("    No se encontraron archivos ML.")
        return

    all_X = []
    all_y = []
    all_timestamps = []
    feature_names_global = None

    for ml_file in ml_files:
        station = ml_file.stem
        dl_file = os.path.join(dl_dir, f"{station}.csv")
        if not os.path.exists(dl_file):
            print(f"    Advertencia: no existe archivo DL para {station}, se omite.")
            continue

        df_ml = pd.read_csv(ml_file, index_col=0, parse_dates=True)
        df_dl = pd.read_csv(dl_file, index_col=0, parse_dates=True)
        if not df_ml.index.equals(df_dl.index):
            print(f"    Advertencia: índices no coinciden para {station}, se omite.")
            continue

        X, y, timestamps, feat = create_windows_with_timestamps(df_ml)
        if X is None:
            continue
        if feature_names_global is None:
            feature_names_global = feat
        else:
            if feat != feature_names_global:
                print(f"    Error: features de {station} no coinciden con las globales. Se omite.")
                continue

        all_X.append(X)
        all_y.append(y)
        all_timestamps.extend(timestamps)

    if not all_X:
        print("    No se generaron ventanas globales.")
        return

    X_global = np.concatenate(all_X, axis=0)
    y_global = np.concatenate(all_y, axis=0)
    # timestamps ya es una lista plana

    # Partición temporal global
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_by_timestamps(
        X_global, y_global, all_timestamps, TRAIN_END, VAL_START, VAL_END, TEST_START
    )

    print(f"    Ventanas globales: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

    # Guardar versión ML (sin normalizar)
    save_split(output_ml_dir, "",   # sin subcarpeta adicional
               X_train, y_train, X_val, y_val, X_test, y_test,
               feature_names_global, scaler_X=None, scaler_y=None)

    # Versión DL normalizada
    if len(X_train) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train, X_val, X_test, y_train, y_val, y_test, feature_names_global
        )
        save_split(output_dl_dir, "",
                   X_train_sc, y_train_sc, X_val_sc, y_val_sc, X_test_sc, y_test_sc,
                   feature_names_global, scaler_X, scaler_y)
    else:
        print("    No hay datos de entrenamiento globales, se omite DL.")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Partición temporal y normalización de ventanas")
    print("=" * 60)

    # 1. Procesar por estación (individual)
    print("\n--- Procesando por estación ---")
    if os.path.exists(INPUT_ML_BY_STATION) and os.path.exists(INPUT_DL_BY_STATION):
        ml_files = {f.stem: f for f in Path(INPUT_ML_BY_STATION).glob("*.csv")}
        dl_files = {f.stem: f for f in Path(INPUT_DL_BY_STATION).glob("*.csv")}
        common = set(ml_files.keys()) & set(dl_files.keys())
        for station in sorted(common):
            process_station(ml_files[station], dl_files[station], station,
                            OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION)
    else:
        print("  Las carpetas de entrada por estación no existen.")

    # 2. Procesar global
    print("\n--- Procesando conjunto global ---")
    if os.path.exists(INPUT_ML_GLOBAL) and os.path.exists(INPUT_DL_GLOBAL):
        process_global(INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                       OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)
    else:
        print("  Las carpetas de entrada global no existen.")

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")
